# Chapter 11: Reinforcement Learning

This notebook accompanies **Chapter 11** of the lecture notes.

> Last lecture the supervision was a label per input — partial, noisy, conflicting, but always pointing at "the right answer". This time the supervision is a single scalar reward, often delayed, often zero, and the agent has to figure out by itself which of its many actions earned it. The chapter formalises the setting as a Markov decision process and walks through the four moves that recur across every RL system: bootstrap a value table, ascend the policy gradient, constrain the step size, balance exploration against exploitation. We do all of this on a tiny Pong-like catcher small enough that **value iteration finds the optimal policy in milliseconds** — giving us a ground truth to measure every subsequent method against.

**Agenda**

🗺️ · 🎯 · 📈 · ✂️ · 🏁

**Take it from here:** 🔭 · 🧠

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones. Everything here is pure NumPy + a tiny SciPy call inside PPO; the entire notebook runs in well under a minute.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import (
    check_bellman_backup, check_value_iteration,
    check_td_target, check_q_learning_update,
    check_compute_returns, check_policy_gradient,
    check_ppo_clipped_objective, check_pick_epsilon_greedy,
)
from viz_helpers import (
    MiniPongEnv, W, H, N_ACTIONS, N_STATES, N_FEATURES, TERMINAL_INDEX,
    ACTION_NAMES, state_index, build_mdp, features, policy_probs,
    rollout_episode, evaluate_policy, greedy_action_from_Q,
    train_q_learning, train_reinforce, train_ppo,
    print_episode, plot_trajectory, plot_value_slice,
    plot_q_vs_vstar, plot_learning_curve, plot_gradient_norms,
    plot_visitation, collect_visitation, epsilon_greedy_using,
    expected_value_under, initial_state_distribution,
)

RNG = np.random.default_rng(0)


## 🗺️ MDP & Bellman

Reinforcement learning starts by writing the world down as a *Markov decision process*: a set of states, a set of actions, a transition function `P(s' | s, a)`, a reward `R(s, a)`, and a discount factor `γ ∈ (0, 1)`. The agent picks an action, the environment moves to a new state and emits a reward, and the loop continues. The Markov property — that the future depends only on the present state — is what makes the next-state distribution a function of `(s, a)` alone, and that locality is what lets the Bellman equation decompose long-horizon return into immediate reward plus discounted future value.

Our world here is **mini-Pong**: a 7×5 grid with a ball that always travels rightward (one column per step, bouncing off top and bottom walls) and a 1-cell-tall paddle that the agent moves up / stay / down. Reward is **0 at every step** until the ball reaches the rightmost column, when the agent receives **+1** if the paddle aligned with the ball (catch) or **−1** otherwise (miss). The state is the 4-tuple `(ball_x, ball_y, ball_vy, paddle_y)`; the action set is `{up, stay, down}`; the discount factor is 0.95.

Write the state count down for what's coming: 6 × 5 × 2 × 5 = **300 non-terminal states**, plus one absorbing terminal. Small enough to enumerate; rich enough that the methods in §🎯 / §📈 / §✂️ each have something real to do.


In [ ]:
env = MiniPongEnv()
print(f'grid                 : {W} columns × {H} rows  (paddle column = {W - 1})')
print(f'state space size     : {N_STATES} (= {(W - 1) * H * 2 * H} non-terminal + 1 terminal)')
print(f'action space         : {N_ACTIONS}  → {ACTION_NAMES}')
print(f'discount factor γ    : 0.95   (sparse terminal reward; γ controls patience)')
print()
s = env.reset(seed=0)
print(f'an example reset state: (bx, by, vy, py) = {s}')


### A random rollout

To see what the environment looks like, take a single episode under a uniformly random policy: pick `up`, `stay`, or `down` with equal probability at every step. Random play catches the ball about 1 time in 5 (any column where the paddle happens to land equal to the ball's row at impact).

The cell below prints every frame of one such episode.  `●` is the ball, `█` is the paddle, `·` is empty.  The right-most column is the paddle column — when the ball reaches it, the episode ends with `+1` or `−1`.


In [ ]:
rng_demo = np.random.default_rng(7)
ep = rollout_episode(env, action_fn=lambda s: int(rng_demo.integers(0, N_ACTIONS)),
                     seed=42)
print_episode(ep['states'], ep['actions'], ep['rewards'],
              header='Random policy, episode trace:')
plot_trajectory(ep['states'],
                title=f"random policy: ball trail + final paddle  (return {ep['total_return']:+.0f})")


**Observe:**
- The ball walks across columns 0 → 6 in exactly 6 steps; every step the paddle gets one chance to move by one row. That bounded horizon is what makes credit assignment tractable here — every reward is at most six steps away from the action that earned it.
- A random paddle catches by accident when its position happens to land on the ball's row. The chapter's central question — can a scalar reward teach what to do? — has very visible answers in this env.
- Notice how the paddle is allowed to move even when the ball is far away. Most early actions look "wasted", but they are not: they set up where the paddle is when it counts.

### One-step Bellman backup

The first move in every RL textbook. Given a value estimate `V(s)` and the MDP's `P` and `R`, the Bellman backup defines the action-value:
```
Q(s, a) = R(s, a) + γ · Σ_{s'} P(s, a, s') · V(s')
```

> Why is this just a *one-step* equation, when the actual return is summed over the entire future? What makes it valid?

<details><summary>Thought</summary>

The Markov property: once we know V at every successor state, the entire infinite sum from `s'` onward has already been folded into `V(s')`. The backup is exact when `V` is exact, and a contraction toward the exact `V` whenever it is not. This is why a global infinite-horizon optimisation reduces to a local fixed-point iteration — the recursion does the heavy lifting.
</details>

Implement `bellman_backup(V, P, R, gamma)`. Inputs: `V` of shape `(N_STATES,)`, `P` of shape `(N_STATES, N_ACTIONS, N_STATES)` with rows summing to 1, `R` of shape `(N_STATES, N_ACTIONS)`, scalar `gamma`. Output: `Q` of shape `(N_STATES, N_ACTIONS)`.


In [ ]:
def bellman_backup(V, P, R, gamma):
    """Single Bellman backup: Q[s, a] = R[s, a] + γ · Σ_s' P[s, a, s'] V[s']."""
    return R + gamma * (P @ V)


check_bellman_backup(bellman_backup)


### Value iteration

Bellman tells us what `Q` should be given a `V`. Value iteration repeats:
```
V_{k+1}(s) = max_a Q_{k+1}(s, a) = max_a [ R(s, a) + γ · Σ_{s'} P(s, a, s') V_k(s') ]
```
starting from `V_0 = 0`. The Bellman operator is a γ-contraction, so this converges geometrically to the unique fixed point `V*`. With `γ = 0.95` and integer rewards in `{−1, 0, +1}`, a hundred iterations are plenty.

Implement `value_iteration(P, R, gamma, n_iters)`. Output: the value function `V*` of shape `(N_STATES,)`. (Hint: reuse the function you just wrote.)


In [ ]:
def value_iteration(P, R, gamma, n_iters):
    """Iterate Bellman backups; take max over actions to update V."""
    V = np.zeros(P.shape[0])
    for _ in range(n_iters):
        Q = bellman_backup(V, P, R, gamma)
        V = Q.max(axis=1)
    return V


check_value_iteration(value_iteration)


In [ ]:
# Build the MDP tensors and run value iteration.
P, R = build_mdp()
print(f'P shape : {P.shape}     (transitions; each P[s, a, :] is a one-hot)')
print(f'R shape : {R.shape}     (rewards)')

V_star = value_iteration(P, R, gamma=0.95, n_iters=200)
Q_star = bellman_backup(V_star, P, R, gamma=0.95)

print()
print(f'V*  range : [{V_star.min():.3f}, {V_star.max():.3f}]')
print(f'V*  averaged over the initial-state distribution : '
      f'{expected_value_under(V_star):.3f}')
print(f'  (this is the optimal expected return — the ceiling every method below is chasing.)')


In [ ]:
# Roll out the optimal policy (greedy w.r.t. Q*) and animate one episode.
def optimal_action(state):
    return int(np.argmax(Q_star[state_index(state)]))

ep = rollout_episode(MiniPongEnv(), optimal_action, seed=42)
print_episode(ep['states'], ep['actions'], ep['rewards'],
              header='Optimal policy (greedy w.r.t. Q*), episode trace:')
plot_trajectory(ep['states'],
                title=f"optimal policy: ball trail + final paddle  (return {ep['total_return']:+.0f})")

print(f"\noptimal policy averaged over 500 random episodes: "
      f"{evaluate_policy(optimal_action, n_episodes=500, seed=0):+.3f}  (≈ +1.0)")


In [ ]:
plot_value_slice(V_star, title='V*  (slice: paddle centred, vy = +1)')


**Observe:**
- The optimal policy catches every ball. From any starting state the paddle has enough time to reach the ball's eventual row, and `V*` averaged over the starting distribution is essentially +1.0.
- Look at the value slice. `V*` is brightest near the central rows (the paddle starts at row 2 and never has to move far) and dimmest at the corners (where the ball ends up far from the paddle's start, and the paddle has to commit early). The structure of "how far does the paddle have to travel before impact" is encoded in the value function's gradient.
- Critically, the optimal policy was found **without ever rolling out a single trajectory**. Value iteration only used `P` and `R`. Every method in the rest of the chapter reverses this: it learns from samples, without access to `P` and `R`.

The gap between value iteration here and Q-learning in §🎯 is the gap between *planning* and *learning* — the central distinction the chapter makes.


## 🎯 Q-learning

Value iteration assumed full access to `P` and `R`. In any non-trivial environment we don't have that — we only have samples: `(s, a, r, s')` tuples drawn from interaction. **Q-learning** is the off-policy, sample-based version of the same Bellman recursion: replace the expectation over `s'` with a single sampled `s'`, and replace the closed-form fixed point with a stochastic update.

The trick — and what makes Q-learning *off-policy* — is that the bootstrap target uses `max_{a'} Q(s', a')` regardless of which action the behaviour rule actually picked next. The agent can therefore learn the optimal policy from data collected under any sufficiently exploratory behaviour, including the past data in a replay buffer.

> Why does Q-learning need to break the correlation between successive `(s_t, s_{t+1})` transitions? What goes wrong if every gradient step uses a tuple drawn from the most recent step?

<details><summary>Thought</summary>

Successive transitions in an episode are heavily correlated — they share a state, share the policy that produced them, share the same trajectory's accidents. SGD on correlated samples behaves like training on a tiny non-i.i.d. subsample, which biases the value estimate toward whatever the agent is currently doing. The replay buffer stores past transitions and lets the update sample over a longer history, *decorrelating* the gradient and making the estimator's variance behave the way SGD theory expects. (DQN's other stabiliser, the target network, addresses a different problem — the regression target moving as fast as the estimator.)
</details>

### TD target

Implement `td_target(r, gamma, q_next_max, done)`. The TD(0) target is `r + γ · max_a' Q(s', a')` — except at terminal transitions, where the bootstrap term must be dropped so we don't bootstrap past the absorbing state.


In [ ]:
def td_target(r, gamma, q_next_max, done):
    """TD(0) bootstrap target.  Drop the future term when done is True."""
    return r + (1.0 - float(done)) * gamma * q_next_max


check_td_target(td_target)


### Q-learning update

Implement `q_learning_update(Q, s, a, r, s_next, done, alpha, gamma)` — one off-policy TD update on a single transition. The rule is:
```
Q[s, a] ← Q[s, a] + α · (target − Q[s, a])
```
where `target = td_target(r, gamma, max_{a'} Q[s_next, a'], done)`. Mutate `Q` in place (or return a new array; the auto-checker accepts either).


In [ ]:
def q_learning_update(Q, s, a, r, s_next, done, alpha, gamma):
    """One off-policy TD update on a single transition.  Returns the updated Q."""
    q_next_max = Q[s_next].max()
    target = td_target(r, gamma, q_next_max, done)
    Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])
    return Q


check_q_learning_update(q_learning_update)


### Train Q-learning and compare to V*

The helper `train_q_learning` runs the full loop: ε-greedy behaviour rule, episodes drawn from `MiniPongEnv`, your `q_learning_update` called on every transition. We train **without** replay first, then re-train **with** replay; the curves diverge because the replay buffer decorrelates the updates.


In [ ]:
GAMMA = 0.95
ALPHA = 0.10
EPS   = 0.10

# Plain on-line Q-learning (no replay): each transition consumed once.
Q_plain, returns_plain = train_q_learning(
    q_learning_update, alpha=ALPHA, gamma=GAMMA, eps=EPS,
    n_episodes=2000, use_replay=False, seed=0,
)

# Q-learning + replay buffer: each step samples a small batch of past transitions.
Q_replay, returns_replay = train_q_learning(
    q_learning_update, alpha=ALPHA, gamma=GAMMA, eps=EPS,
    n_episodes=2000, use_replay=True, buffer_size=2000, batch_size=8, seed=0,
)

print('Average return on 500 fresh episodes under each greedy policy:')
print(f'  greedy(Q-learning, no replay)   : '
      f'{evaluate_policy(greedy_action_from_Q(Q_plain), n_episodes=500, seed=0):+.3f}')
print(f'  greedy(Q-learning, with replay) : '
      f'{evaluate_policy(greedy_action_from_Q(Q_replay), n_episodes=500, seed=0):+.3f}')
print(f'  optimal (V*)                    : '
      f'{evaluate_policy(optimal_action, n_episodes=500, seed=0):+.3f}')


In [ ]:
plot_learning_curve({
    'no replay'   : returns_plain,
    'with replay' : returns_replay,
}, title='Q-learning: episode return per training episode', smooth_k=50)

plot_q_vs_vstar(Q_replay, V_star,
                title='Q-learning (with replay) vs V*  — every dot is one state')


**Observe:**
- The episode-return curve climbs from random play (≈ −0.6) toward the optimal +1, smoothed over a sliding window because per-episode returns are ±1 and look noisy raw.
- The replay buffer makes a visible difference at this state-space size: it tightens the variance of updates and drives the Q-table closer to V* (the scatter plot's diagonal).
- The few off-diagonal dots in the Q-vs-V* plot are states the random behaviour rule visited rarely. The lecture's claim that "exploration matters" has a concrete face here — those are the states where the agent's value estimate has not yet converged because the data has not yet covered them.
- Crucially Q-learning never used `P` or `R` directly. It learned from samples alone, by **bootstrapping** every estimate against the next one. That is the move value iteration generalises to.


## 📈 Policy gradient

Q-learning learned a value table; the policy was always a side effect (greedy w.r.t. `Q`). Policy-gradient methods flip that around: they parameterise the policy directly and ascend the gradient of expected return.

The policy here is a **linear softmax** over three hand-crafted features of the state:

| feature | meaning |
|---------|---------|
| `(py − by) / (H − 1)` | current paddle-ball vertical offset |
| `(py − impact_y) / (H − 1)` | offset to the *predicted impact y* (with bouncing) |
| `1.0` | bias |

The second feature is the engineered signal a linear policy alone cannot synthesise: predicting the impact y is *piecewise non-linear* in `(by, vy, bx)` because the ball reflects off the top and bottom walls.  Pre-computing it lets the policy combine immediate tracking with anticipation in a single weighted sum.

So `θ` has shape `(N_ACTIONS, N_FEATURES) = (3, 3)` — only nine parameters. `π(a|s) = softmax(θ · φ(s))_a`.  Training is REINFORCE: collect an episode, compute the discounted return at every timestep, then nudge `θ` in the direction of `Σ_t advantage_t · ∇θ log π(a_t | s_t)`.

> The gradient of the expected return depends on the *whole future* of every action.  Why is REINFORCE able to estimate it from a single rollout?

<details><summary>Thought</summary>

The score-function trick rewrites `∇_θ E_π[R] = E_π[ R · ∇_θ log π(a|s) ]`. Inside the expectation, every action's gradient term is multiplied by *its* future return — past rewards drop out by the policy's no-look-ahead property. A single rollout gives one sample of that expectation; many rollouts give a Monte-Carlo estimate. The variance is high (single trajectories are noisy), which is why REINFORCE benefits from a baseline that does not change the gradient in expectation but reduces the per-sample variance.
</details>

### Discounted returns

Implement `compute_returns(rewards, gamma)`. Given a rewards sequence of length `T`, return a length-`T` array where the `t`-th entry is `G_t = Σ_{k ≥ t} γ^{k−t} r_k`.


In [ ]:
def compute_returns(rewards, gamma):
    """Per-timestep discounted return G_t = Σ_{k≥t} γ^{k−t} r_k."""
    T = len(rewards)
    returns = np.zeros(T)
    G = 0.0
    for t in reversed(range(T)):
        G = rewards[t] + gamma * G
        returns[t] = G
    return returns


check_compute_returns(compute_returns)


### Policy gradient

Implement `policy_gradient(theta, phis, actions, advantages)`. The gradient of `log π(a | s)` w.r.t. `θ` for a softmax policy is:
```
grad[k, :] = (1[k == a] − π(k | s)) · φ(s)
```
That is, *every* action row of `θ` gets a contribution — the chosen one positive, the rest pulled down by their probability mass. Sum over timesteps weighted by advantages, and you have the policy gradient:
```
g = Σ_t advantage_t · grad_log_π(a_t | s_t)
```

Inputs: `theta` of shape `(N_ACTIONS, N_FEATURES)`, `phis` a list of `(N_FEATURES,)` vectors, `actions` a list of int actions, `advantages` a list/array of floats.  Output: gradient of shape `(N_ACTIONS, N_FEATURES)`.


In [ ]:
def policy_gradient(theta, phis, actions, advantages):
    """Sum of advantage-weighted ∇log π over a trajectory."""
    n_actions, n_features = theta.shape
    grad = np.zeros_like(theta)
    for phi, a, adv in zip(phis, actions, advantages):
        probs = policy_probs(theta, phi)
        indicator = np.zeros(n_actions); indicator[int(a)] = 1.0
        grad += float(adv) * np.outer(indicator - probs, phi)
    return grad


check_policy_gradient(policy_gradient)


### Train REINFORCE — and add a baseline

Two runs: the first uses raw returns as the advantage signal; the second subtracts a **running-average baseline** so that the per-step advantage is the deviation from the policy's typical performance. The baseline does not change the gradient in expectation (it adds a constant, which integrates to zero against `∇log π`), but it sharply reduces the variance of the per-episode estimate.


In [ ]:
theta_pg, returns_pg, grad_norms_pg = train_reinforce(
    compute_returns, policy_gradient,
    lr=0.10, gamma=GAMMA, n_episodes=1200, baseline=False, seed=0,
)
theta_pg_b, returns_pg_b, grad_norms_pg_b = train_reinforce(
    compute_returns, policy_gradient,
    lr=0.10, gamma=GAMMA, n_episodes=1200, baseline=True, seed=0,
)

def policy_action(theta):
    return lambda s: int(np.argmax(theta @ features(s)))

print('Average return on 500 fresh episodes under each policy:')
print(f'  REINFORCE                 : '
      f'{evaluate_policy(policy_action(theta_pg), n_episodes=500, seed=0):+.3f}')
print(f'  REINFORCE + baseline      : '
      f'{evaluate_policy(policy_action(theta_pg_b), n_episodes=500, seed=0):+.3f}')
print(f'  optimal                   : '
      f'{evaluate_policy(optimal_action, n_episodes=500, seed=0):+.3f}')


In [ ]:
plot_learning_curve({
    'REINFORCE'              : returns_pg,
    'REINFORCE + baseline'   : returns_pg_b,
}, title='Policy gradient: episode return', smooth_k=30)

plot_gradient_norms({
    'no baseline'      : grad_norms_pg,
    'with baseline'    : grad_norms_pg_b,
}, title='Gradient-norm trajectories — baseline reduces variance')


**Observe:**
- The baselined run converges to the optimal +1 plateau; the no-baseline run gets stuck around −0.6 — barely better than random. The chapter's claim "a state-dependent baseline leaves the gradient unbiased and shrinks its variance" has a dramatic footprint here: with returns averaging strongly negative under random play, every gradient update without a baseline pushes mass *away* from whichever action was taken, regardless of whether that action was good. The baseline centres the signal so the gradient pushes mass *toward* better-than-average actions and *away* from worse-than-average ones — exactly the directionality the Monte-Carlo estimator was supposed to provide.
- The gradient-norm plot tells the same story without averaging: the baselined run has dramatically smaller `||∇θ J||` after the early episodes, because we subtract off the bulk of the return that does not vary with the action.
- Notice this policy uses **9 parameters total**, which is far fewer than the 900-cell tabular `Q` table in §🎯. Linear features generalise across states — the same 9 weights cover every `(by, vy, py)` triple, *because* the impact-y feature does the bouncing arithmetic for them. This is the bridge to neural-network policies: replace the hand-crafted feature with a small MLP that learns the non-linearity end-to-end and you have actor-critic / DQN-policy-head territory.


## ✂️ PPO

REINFORCE is on-policy: every gradient step uses data drawn under the *current* `θ`. Take too aggressive a step and the new `θ` distributes mass over actions the old data has nothing to say about, and the next gradient estimate is noise.

The principled fix — **trust-region policy optimisation** — constrains the KL between successive policies. PPO replaces that constraint with a simpler **clipped surrogate** that is cheap to optimise with first-order methods. For each timestep with importance ratio `r = π_θ(a|s) / π_{θ_old}(a|s)` and advantage `A`, the surrogate objective is:
```
L = E_t [ min( r_t · A_t,   clip(r_t, 1 − ε, 1 + ε) · A_t ) ]
```
The `min` is what makes the bound *pessimistic*: when clipping would help the loss (advantage and ratio in the wrong direction), the unclipped term wins; when it would hurt (advantage and ratio aligned, ratio outside the trust region), the clipped term caps the gain.

> The clip range ε is the practical knob.  Why does the surrogate use `min(unclipped, clipped)` rather than just `clipped`?

<details><summary>Thought</summary>

If we only used `clipped`, the gradient at ratios already outside `[1 − ε, 1 + ε]` would be zero — but only in the direction that *helps* the policy. We also need to penalise moves that slip outside the trust region in the *unhelpful* direction (large ratio with negative advantage, or small ratio with positive advantage). The unclipped branch contributes the gradient there. Taking the elementwise `min` keeps the unclipped term exactly when it tightens the surrogate, and the clipped term otherwise.
</details>

### Clipped surrogate

Implement `ppo_clipped_objective(ratios, advantages, eps_clip)`. Returns the **scalar mean** of `min(r · A, clip(r, 1 − ε, 1 + ε) · A)` — to be **maximised** by the training loop.


In [ ]:
def ppo_clipped_objective(ratios, advantages, eps_clip):
    """Mean clipped surrogate. Maximise this; training negates it internally."""
    clipped = np.clip(ratios, 1.0 - eps_clip, 1.0 + eps_clip)
    return float(np.mean(np.minimum(ratios * advantages, clipped * advantages)))


check_ppo_clipped_objective(ppo_clipped_objective)


### Sweep the clip range

We train PPO with three values of `ε ∈ {0.05, 0.2, 0.5}` and compare the iteration-return curves.  Small `ε` is the tight trust region — slow but very stable; large `ε` is more aggressive — faster early gains but riskier later.  The default in published PPO implementations is 0.2.


In [ ]:
ppo_curves = {}
ppo_thetas  = {}
for eps_clip in (0.05, 0.20, 0.50):
    theta_ppo, returns_ppo = train_ppo(
        ppo_clipped_objective, compute_returns,
        eps_clip=eps_clip, gamma=GAMMA,
        n_iters=80, batch_episodes=10, n_epochs=3, seed=0,
    )
    ppo_curves[f'ε = {eps_clip}'] = returns_ppo
    ppo_thetas[eps_clip] = theta_ppo
    print(f"PPO ε={eps_clip}: final greedy return = "
          f"{evaluate_policy(policy_action(theta_ppo), n_episodes=500, seed=0):+.3f}")

plot_learning_curve(ppo_curves, title='PPO: average batch return per outer iteration',
                    smooth_k=5)


**Observe:**
- All three clip ranges eventually converge — on this small env there is no value of `ε` that catastrophically destabilises the policy. But the *shape* of the curve differs: tight `ε = 0.05` makes monotone, slow progress; loose `ε = 0.50` jumps fast then plateaus.
- The chapter's framing — "the clip range becomes the practical knob that trades stability for sample efficiency" — has a concrete picture here. On a harder problem (e.g. continuous control, language-model RLHF) the wrong `ε` would not just be slower, it would actively destabilise; the trust region is what keeps the off-by-policy update *safe*.
- Notice that PPO never used the env's `P` and `R` either: it only used sample trajectories, like Q-learning and REINFORCE. The three forces the chapter named — credit assignment (Bellman / returns), stability (replay / trust regions), exploration (still upcoming) — show up here as the three things you have to engineer around.


### 🏁 Recap

**What we did:**
- 🗺️ Wrote mini-Pong as a Markov decision process and ran value iteration on the full `(P, R)` tensor pair to find the exact `V*` — the planning-from-the-model baseline that every learning-from-samples method below was measured against.
- 🎯 Implemented Q-learning's TD target and update rule, trained with ε-greedy behaviour, and showed the replay buffer closing the gap to `V*`. Sample-based, off-policy, bootstrapped — the moves that scale up to DQN.
- 📈 Switched to a 9-parameter linear softmax policy and trained REINFORCE directly. Adding a running-average baseline turned per-step *return* into per-step *advantage* and visibly shrank the gradient norm.
- ✂️ Constrained the policy step with PPO's clipped surrogate. Small ε is slow and safe; large ε is fast and risky.

**Key takeaways:**
- The Bellman equation is the engine room. Value iteration, Q-learning, and the value-function baseline used inside actor-critic are all the same recursion at different sample budgets and with different access to `P`.
- Credit assignment (Bellman backups, discounted returns), stability (replay buffers, trust regions), and exploration are the three forces that shape every method. Each method in this notebook is a particular trade-off among the three.
- A single scalar reward — sparse, delayed, sometimes silent — was enough to teach an agent how to act, with no teacher ever specifying the right action. The fundamental question the chapter posed has a concrete affirmative answer in this notebook.

The take-it-from-here below picks up the third force, exploration, on a sparse-reward variant of the env where the failure mode of "greedy and stuck at random" becomes visible. The bridge to function-approximation RL (DQN and beyond) is sketched in the second extension, where the tabular `Q` becomes a linear or neural function of the state.

The next chapter picks up where this one leaves off: when the reward is no longer a fixed function of the environment but a learned model of human preferences, and the same machinery has to navigate that.


## Take It from Here, Next Steps

Two optional extensions that each open a door onto a research thread of their own. Work through them at your own pace after the session.

### 🔭 Exploration

Every method above used ε-greedy or a stochastic policy as the *behaviour rule* — the source of trajectories. With reward at every miss as well as every catch, the env was informative enough that even a 10% random policy explored adequately. Real RL problems are rarely that generous.

A **sparse-reward variant** of mini-Pong sets the miss reward to 0 instead of −1. Now the only training signal is the +1 for catching, and a greedy policy that misses on its first random initialisation receives **no signal at all** to update from. The behaviour rule has to take the agent off-policy enough to sometimes catch the ball by accident, so the value table can latch on to that signal and sharpen.

> Why does ε-greedy with `ε = 0` fail catastrophically here, when it would still work (slowly) on the dense-reward env above?

<details><summary>Thought</summary>

With the dense reward, every miss returns −1 — the agent always learns *something*, including "this whole region of state space is bad". The greedy update has gradient even on losing trajectories. With the sparse reward, missing returns 0; the gradient on every miss is zero; the Q-table never updates. The policy stays at its initialisation forever unless some random behaviour eventually catches a ball and writes a positive value somewhere. ε-greedy, entropy bonuses, and stochastic policies are the three tools the chapter listed for forcing that initial successful sample.
</details>

Implement `pick_epsilon_greedy(Q_row, eps, rng)`. Input: a length-`N_ACTIONS` row of Q-values for one state, scalar `eps`, a NumPy `rng`. Output: an integer action in `[0, N_ACTIONS)`. The rule is: with probability `eps` pick uniformly at random; otherwise return the argmax.


In [ ]:
def pick_epsilon_greedy(Q_row, eps, rng):
    """ε-greedy action: random with prob eps, argmax otherwise."""
    if rng.random() < eps:
        return int(rng.integers(0, len(Q_row)))
    return int(np.argmax(Q_row))


check_pick_epsilon_greedy(pick_epsilon_greedy)


### Compare exploration strategies on the sparse-reward env

We train Q-learning under three behaviour rules: pure greedy (`ε = 0`), mild exploration (`ε = 0.10`), and aggressive exploration (`ε = 0.40`). All three use the **sparse** reward variant where misses return 0. The state-visitation heatmaps tell you what each rule actually saw during training, and the per-rule average return tells you how well it learned.


In [ ]:
SPARSE_KW = dict(alpha=0.10, gamma=0.95, n_episodes=2000, sparse=True, seed=0)

# Greedy: eps = 0 means no exploration; updates only happen if the agent
# stumbles into a +1 by chance under its (deterministic) initial Q.
Q_g0,  _ = train_q_learning(q_learning_update, eps=0.00, **SPARSE_KW)
Q_g10, _ = train_q_learning(q_learning_update, eps=0.10, **SPARSE_KW)
Q_g40, _ = train_q_learning(q_learning_update, eps=0.40, **SPARSE_KW)

print('Average return on 500 fresh sparse-reward episodes under each greedy policy:')
for tag, Q in (('eps = 0.00 (pure greedy)', Q_g0),
               ('eps = 0.10 (mild)',         Q_g10),
               ('eps = 0.40 (aggressive)',   Q_g40)):
    avg = evaluate_policy(greedy_action_from_Q(Q), n_episodes=500, seed=0, sparse=True)
    print(f'  {tag:<26}: {avg:+.3f}')


In [ ]:
# State visitation under each ε-greedy behaviour rule, on sparse rollouts.
RNG_VIS = np.random.default_rng(0)
visits_g0,  ret_g0  = collect_visitation(
    epsilon_greedy_using(pick_epsilon_greedy, Q_g0, 0.00, RNG_VIS),
    n_episodes=300, seed=0, sparse=True,
)
RNG_VIS = np.random.default_rng(0)
visits_g10, ret_g10 = collect_visitation(
    epsilon_greedy_using(pick_epsilon_greedy, Q_g10, 0.10, RNG_VIS),
    n_episodes=300, seed=0, sparse=True,
)
RNG_VIS = np.random.default_rng(0)
visits_g40, ret_g40 = collect_visitation(
    epsilon_greedy_using(pick_epsilon_greedy, Q_g40, 0.40, RNG_VIS),
    n_episodes=300, seed=0, sparse=True,
)
plot_visitation(visits_g0,  title='visitation, ε = 0.00  (pure greedy)')
plot_visitation(visits_g10, title='visitation, ε = 0.10')
plot_visitation(visits_g40, title='visitation, ε = 0.40  (aggressive)')


**Observe:**
- Pure greedy on the sparse-reward variant **fails to learn**. The visitation heatmap collapses to a thin ridge — the agent visits the same states over and over because its (frozen) policy is deterministic. It almost never sees a +1 reward, so its Q-table cannot update and it cannot find the policy that catches.
- Mild ε (0.10) is enough to bootstrap. The visitation spreads across a wider band of states; some of those visits land on catches, the +1 propagates back through the Bellman backup, and the policy steers itself toward the optimal +1 plateau.
- Aggressive ε (0.40) explores even more widely (the visitation heatmap is the most uniform), but the *behaviour* return drops because the agent keeps making random moves at evaluation. The greedy-extracted policy is still good, though, because the value table has been thoroughly fitted.
- This is the classical ε-greedy trade-off: exploration buys you the data, exploitation buys you the reward. Entropy bonuses and stochastic policies (REINFORCE / PPO) are the two other instruments that show up everywhere; better exploration — count-based, intrinsic reward, curiosity — is its own active subfield.


### 🧠 From tabular to function approximation

Every method in this notebook used a 1500-cell `Q[state, action]` table or a 9-weight linear softmax policy. Tabular methods scale linearly in the number of states, which is fine on a 7×5 grid and ruinous anywhere else — the moment the input is high-dimensional (raw pixels, language tokens, sensor streams) you cannot enumerate states, let alone visit them all enough times to bootstrap.

The replacement is *function approximation*. Replace `Q[s, a]` with `Q(s, a; w)` — a parametric function of the state, with parameters `w` shared across all states. The Bellman backup becomes a regression: minimise `(target − Q(s, a; w))²` with `target = r + γ · max_{a'} Q(s', a'; w)`. This is **DQN**: the function approximator is a neural network, the data is a replay buffer of past transitions, and the target uses a *target network* whose parameters update only periodically so the regression target stops moving as fast as the estimator can chase it.

The same generalisation applies to the policy: `π(a | s; θ)` becomes a neural network instead of a linear softmax, and you have **actor-critic**. PPO with neural-network policies and value functions is the workhorse of robotics, language-model RLHF, and most game-playing systems shipped in the last decade.

Concretely, you would:

1. Replace the Q-table with a small MLP `Q(s; w) → R^{N_ACTIONS}` taking the state's features (or a learned embedding) as input.
2. Sample minibatches from a replay buffer of past `(s, a, r, s')` transitions.
3. Compute `target = r + γ · max_{a'} Q(s'; w_target)` using a *target network* whose weights are a delayed copy of `w`.
4. Update `w` to minimise the squared TD error with stochastic gradient descent.
5. Periodically (every C steps) copy `w → w_target`.

The same three forces you saw above — credit assignment, stability, exploration — shape every design choice. Replay decorrelates updates; target networks freeze the regression target; ε-greedy or entropy bonuses keep the data exploratory. The chapter's framing was deliberately abstract enough that this neural extension is just a different choice of representation; the storyline is the same.

This is one of the active research frontiers — model-based methods, world models, offline RL, and reward-modelling for language models all build on the same machinery you just wrote, with the *function* swapped for one with billions of parameters and a feature extractor learned end-to-end.
